In [ ]:
## Goal of the pipeline is to conduct inference for the MicroLane Experiment
## We evaluate four different model categories in three different datasets
## We also evaluate for different environmental conditions using filters
## And the data generated by this experiment will help us compare for which data works
## We evaluate if lane detection models work for 1/10 RC car conditions as well

### Defualt Imports and Library Imports from the Core Package

In [ ]:
import random
from tqdm.notebook import tqdm

In [ ]:
from microlane.utils.load_config import load_config
from microlane.utils.experiment import Experiment
from microlane.utils.create_settings import create_settings

from microlane.schemas.sample import Sequence
from microlane.schemas.prediction import Prediction

from microlane.augmentation.augmentor import Augmentor

from microlane.datasets.microlane import MicroLane
from microlane.datasets.tusimple import TuSimple

from microlane.models.rld_a.model import RLD as RLD_A
from microlane.models.rld_b.model import RLD as RLD_B

### Loading the Configuration File

In [ ]:
# Importig and Printing the Configuration file
config = load_config()
print(config)

EXPERIMENT_DIRECTORY=config.experiment.experiment_directory
SAMPLE_LENGTH=config.experiment.sample_length
INFERENCE_IMAGE_SAMPLING_NUMBER=config.experiment.inference_image_sampling_number
MODEL=config.experiment.model.lower()
DATASET=config.experiment.dataset.lower()
AUGMENTATION=config.experiment.augmentation.lower()

Config(experiment=Experiment(experiment_directory=PosixPath('/home/suyog/desktop/projects/microlane/results/experiment/microlane/rld_b/camera_shake'), testing_directory=PosixPath('/home/suyog/desktop/projects/microlane/results/testing'), sample_length=450, inference_image_sampling_number=10, model='rld_b', dataset='microlane', augmentation='camera_shake'), constants=Constants(h_samples=[160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 500, 510, 520, 530, 540, 550, 560, 570, 580, 590, 600, 610, 620, 630, 640, 650, 660, 670, 680, 690, 700, 710], default_port=8000, colors=Colors(lane_colors=[[255, 0, 0], [0, 255, 0], [0, 0, 255], [0, 255, 255]])), datasets=Datasets(tusimple=Dataset(name='tusimple', path=PosixPath('/home/suyog/assets/datasets/TuSimple/TUSimple'), annotation_file=PosixPath('/home/suyog/assets/datasets/TuSimple/test_label_new.json'), dimensions=[720, 1280]

### Loading the dataset on the Computer Memory

In [ ]:
if DATASET == "tusimple":
    dataset = TuSimple(
        annotation_file_path=config.datasets.tusimple.annotation_file,
        folder_path=config.datasets.tusimple.path
    )
elif DATASET == "microlane":
    dataset = MicroLane(
        annotation_file_path=config.datasets.microlane.annotation_file,
        folder_path=config.datasets.microlane.path
    )
elif DATASET == "modified_microlane":
    dataset = MicroLane(
        annotation_file_path=config.datasets.modified_microlane.annotation_file,
        folder_path=config.datasets.modified_microlane.path
    )
else:
    raise ValueError(f"Dataset Must be tusimple, microlane or modified microlane, [Dataset = {DATASET}]")

#### Generating random Sample Indexs to Visualize

In [ ]:
# Random Sample Index to Visualize during the Process
numbers = [random.randint(0, SAMPLE_LENGTH - 1) for _ in range(INFERENCE_IMAGE_SAMPLING_NUMBER)]
print(numbers)

[195, 229, 336, 104, 213, 342, 117, 159, 369, 429]


### Loading the Model in the System to Conduct Predictions
Creates a docker container with the model running on it, uses fastapi api endpoint to connect to that docker container

In [ ]:
if MODEL == "rld_a":
    model = RLD_A(
    )
elif MODEL == "rld_b":
    model = RLD_B(
    )
else:
    raise ValueError(f"Model must be either rld_a or rld_b, [Model = {MODEL}]")

Waiting for container to be ready on port 8004...
Container is ready.


### Initializing the Experiment Class and Storing Settings in the Output Directory

In [ ]:
experiment = Experiment(EXPERIMENT_DIRECTORY)

settings = create_settings(
    inference_vis_number=INFERENCE_IMAGE_SAMPLING_NUMBER,
    sample_number=SAMPLE_LENGTH,
    model=MODEL,
    dataset=DATASET,
    augmentation_type=AUGMENTATION,
    output_path=str(experiment.folder),
)

### Initialize the augmentor for augmenting the images

In [ ]:
# Initializng the augmentor to apply augmentations like blur, motion_blur, lightin_b, etc

auggmentor = Augmentor()

### Looping through the Samples to Generate Predictions

In [ ]:
for index, testing_example in enumerate(tqdm(dataset.load_sequences(number=SAMPLE_LENGTH), total=SAMPLE_LENGTH)):        

    augmented_sequence = auggmentor.apply_preset_to_sequence(testing_example, AUGMENTATION)
    
    response: Prediction = model.predict(augmented_sequence)
    
    experiment.store_prediction(response)
    
    if index in numbers:
        
        experiment.visualize_prediction(response, show=False)
        
    if (index % 100 == 0) and (index != 0):
        
        print(f"Routine Container Restart for Addressing Memory Leak [{int(index/100)}]")
        
        model.restart(
            
            model.container_id,
            getattr(config.models, MODEL),
        )

  0%|          | 0/450 [00:00<?, ?it/s]

Routine Container Restart for Addressing Memory Leak [1]
Waiting for container to be ready on port 8004...
Container is ready.
Routine Container Restart for Addressing Memory Leak [2]
Waiting for container to be ready on port 8004...
Container is ready.
Routine Container Restart for Addressing Memory Leak [3]
Waiting for container to be ready on port 8004...
Container is ready.
Routine Container Restart for Addressing Memory Leak [4]
Waiting for container to be ready on port 8004...
Container is ready.
